In [ ]:
import numpy as np
import math
from pathlib import Path

In [ ]:
%matplotlib widget

In [ ]:
data = np.load('D1_Manifold_new.npy', allow_pickle=True)
obj = data.item() 
x1 = obj['x1']
v2 = obj['v2']
T = obj['T']
C = obj['C']
mu = obj['mu']
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen
from scipy.integrate import solve_ivp

In [ ]:
def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt

In [ ]:
fig_man, ax_man = plt.subplots(1)
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, x1.size, 5):
    #print(i)
    p0 = np.array([x1[i], 0, 0, v2[i]])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1[i], res, '.k')
    res_list.append(res)
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
np.argmax(res_list)

In [ ]:
res_list = []
for i in range(x1.size):
    p0 = np.array([x1[i], 0, 0, v2[i]])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-8,
                  rtol=1e-8)
    res = np.linalg.norm(p.y[:,-1] - p0)
    res_list.append(res)
weights = np.array(res_list)
weights_norm = (1/weights)/np.max(1/weights)

In [ ]:
plt.plot(x1, weights_norm)
plt.grid()
plt.xlabel('x1')
plt.ylabel('v2')
#plt.legend()
plt.show()

In [ ]:
# %%
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

In [ ]:
# Считаем первую и вторую производные по a
a = x1
dv2_da = np.gradient(v2, a)
d2v2_da2 = np.gradient(dv2_da, a)

def find_zeros(x, y):
    """Находит x-координаты нулей функции y(x) методом линейной интерполяции."""
    sign_change = np.where(np.diff(np.sign(y)))[0]
    zeros = []
    for i in sign_change:
        x0, x1 = x[i], x[i+1]
        y0, y1 = y[i], y[i+1]
        # Линейная интерполяция для точного положения нуля
        zero_x = x0 - y0 * (x1 - x0) / (y1 - y0)
        zeros.append(zero_x)
    return np.array(zeros)

# Находим нули производных
zeros_first = find_zeros(a, dv2_da)
zeros_second = find_zeros(a, d2v2_da2)

# Для отображения на графике получаем соответствующие значения v2 в этих точках
v2_at_zeros_first = np.interp(zeros_first, a, v2)
v2_at_zeros_second = np.interp(zeros_second, a, v2)

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(a, v2, label="v2", color='blue')
plt.scatter(zeros_first, v2_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second, v2_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('v2')
plt.legend()
plt.title("График функции v2 и нули её производных")
plt.grid(True)
plt.show()

In [ ]:
zeros_first

In [ ]:
x1

In [ ]:
model = PySRRegressor(
        niterations=5000,  # < Increase me for better results
        binary_operators=["*", "+", "-","/","^"],
        unary_operators=[
            'exp',
            'cos',
            'sin',
            'log',
        ],
        #extra_sympy_mappings={"pimult": lambda x: np.pi*x},
        #nested_constraints={"cos": {"cos": 0, "sin": 0},"sin": {"cos": 0, "sin": 0}},
        constraints={'^': (-1, 1)},
        #nested_constraints={"exp": {"exp": 0}},
        #nested_constraints={"exp": {"exp": 0},"log": {"log": 0}},
        nested_constraints={"exp": {"exp": 0}, "^": {"^": 0},"log": {"log": 0},"cos":{"cos": 0,"sin": 0, "exp": 0,"log": 0},"sin":{"cos": 0,"sin": 0,"exp": 0,"log": 0}},
        populations=36,
        verbosity=0,
        #batching=True,
        warm_start = False,
        #denoise=True
    )

In [ ]:
input_values = np.expand_dims(x1.flatten(),-1)
target_values = np.expand_dims(v2.flatten(),-1)
model.fit(input_values, target_values,
        variable_names=["x"])

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import x

In [ ]:
N=None

test_preds = model.predict(input_values,N)
test_trues = target_values.reshape(test_preds.shape)
MSE = np.square(np.subtract(test_trues,test_preds)).mean() 
s = model.sympy(N).simplify()
display(s)
RMSE = np.sqrt(MSE)
print(RMSE)

print('Python Code:')
f = lambdastr(x, s)
print(f)
plt.plot(input_values, test_trues)
plt.plot(input_values, test_preds,'--')
plt.grid()
plt.show()

In [ ]:
np.max(x1)

In [ ]:
x1_fun = lambda x: x
x2_fun = lambda x: 0

def v1_fun(x):
    return 0
def v2_fun(x):
    #y = ((x*(x*(x**17.564913 - 0.094613895)*(math.log(x) + 0.202557096905805)*math.sin(x + x**16.484608 + math.log(x)) - math.sin(x) + 0.7687056)*math.log(x) - 2.2717886*x + 0.4104162*math.log(x) + 2.287544)/(x*math.log(x))) #3.5e-04
    if x >= 0.0864372 and x < 0.225:
        y = ((x*(-x + 0.75904304*math.log(x) + 4.5340606e-5*math.sin((x - math.exp((math.log(x) + 0.62599350956354)/x))/x**2) - 0.91062963) + 0.03991896*math.sin(math.log(x)) - 0.113537416)/x) #1.0e-06 #(0.0864372619790192,0.2247207032753864)
    elif x >= 0.225 and x < 0.77609249:
        y = (x**(-0.42583352)*((x - 0.39050043)*math.atan(x*(0.028901706*math.sin(math.exp(2.38506701907145*x**2) - 0.64761937/x) - 1.0)) - 1.6003051)) #1.1e-04 #(0.0864372619790192,0.7841749767104571)
    elif x >= 0.77609249 and x < 0.97:
        #y = (x*(-x + ((0.0041173585 - 0.000111737405/math.log(x))/math.log(x) - 3.1406267)/math.sin(x) + 2.6212773)) #2.6e-04 #(0.7341749767104571,0.969870790795701)
        y = (-1.3297951 + 0.08821651/math.log(math.atan(x)) + 0.6585475/(92.36741*math.log(x) + 0.802669) - 0.38527542/x) #8.1e-05 #(0.7341749767104571,0.969870790795701)
    else:
        y = 0
    return y
def T_fun(x):
    #y = (-0.3513668*math.tanh(0.091483041197784*x**3.2208656/math.log(x + 0.02436921) - 0.76125264*x**4.2208656) + 6.2982473 - 0.051352303/math.log(x)) #6.4e-4
    y = ((x**5.910087 + x**19.891893 + 0.16254957*math.atan(math.atan(1.32649096850317*x**1.7503009)) + 6.317418)/math.cos(x**34.099598)) #9.8e-04 #(0.0864372619790192,0.969870790795701)
    return y
#x_range = np.linspace(0.0865, 0.965, 100)

#x_range = np.linspace(0.0865, 0.965, 10)
x_range = np.array([0.0865,0.5,0.965])
fig_man, ax_man = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], 'k')
    #ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
#ax_man.plot(L1, 0, '*k')
#ax_man.plot(L2, 0, '*k')
#ax_man.plot(L3, 0, '*k')
ax_man.grid()
#ax_res.grid()

ax_man.set_xlabel('x, безразм.', fontsize=11)
ax_man.set_ylabel('y, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'D1_orbits.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')


In [ ]:

x_range = np.linspace(np.min(x1), np.max(x1), 100)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'D1_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:

a = x1
trues = v2
preds = [v2_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x$, безразм.')
ax.set_ylabel(r'$v_y$, безразм.')
ax.legend(loc='upper left')

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'D1_vy.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
a = x1
trues = T
preds = [T_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x$, безразм.')
ax.set_ylabel(r'$T$, безразм.')
ax.legend()

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'D1_T.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((x*(-x + 0.75904304*math.log(x) + 4.5340606e-5*math.sin((x - math.exp((math.log(x) + 0.62599350956354)/x))/x**2) - 0.91062963) + 0.03991896*math.sin(math.log(x)) - 0.113537416)/x) #1.0e-06"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=15, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(x**(-0.42583352)*((x - 0.39050043)*math.atan(x*(0.028901706*math.sin(math.exp(2.38506701907145*x**2) - 0.64761937/x) - 1.0)) - 1.6003051)) #1.1e-04"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=16, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-1.3297951 + 0.08821651/math.log(math.atan(x)) + 0.6585475/(92.36741*math.log(x) + 0.802669) - 0.38527542/x) #8.1e-05"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=17, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((x**5.910087 + x**19.891893 + 0.16254957*math.atan(math.atan(1.32649096850317*x**1.7503009)) + 6.317418)/math.cos(x**34.099598)) #9.8e-04"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=9, litera=litera)
#latex = latex.replace('x','\mu')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)